# JEPA intuition — predict embeddings, not pixels

Companion to `examples/jepa_basics/01_train_jepa_mnist.py`. This notebook builds the
intuition behind the generative-JEPA vertical slice:

1. **Masking** — split MNIST patches into a *context* set and a *target* set.
2. **Prediction in latent space** — the JEPA predicts the *embeddings* of the target
   patches from the context, never the pixels. An EMA target encoder supplies the targets.
3. **Collapse** — the failure mode to watch: every input mapping to the same embedding.
   We track effective rank / feature std.
4. **Sampling** — once frozen, a flow prior over the pooled latent + a decoder turn the
   representation learner into a sampleable generative model.

Run the example scripts for the full training; here we just visualize the pieces.

In [ ]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1] / 'src'))
import torch
from ssllab.data.mnist import get_mnist_dataloaders, patchify, N_TOKENS, TOKEN_DIM, GRID
from ssllab.jepa.masking import sample_block_masks
from ssllab.utils import set_seed, get_device
set_seed(0); device = get_device(); print('device', device, '| tokens', N_TOKENS, '| token_dim', TOKEN_DIM)

## 1. Context / target masking on the 4×4 patch grid

In [ ]:
import matplotlib.pyplot as plt
train_loader, _ = get_mnist_dataloaders(batch_size=8)
images, _ = next(iter(train_loader))
ctx_idx, tgt_idx = sample_block_masks(N_TOKENS, n_target=4)
print('context patches:', ctx_idx.tolist())
print('target patches: ', tgt_idx.tolist())

# Visualize the mask over one digit (target patches blacked out = what the model must predict).
img = images[0, 0].clone()
p = 7
for t in tgt_idx.tolist():
    r, c = divmod(t, GRID)
    img[r*p:(r+1)*p, c*p:(c+1)*p] = 0.0
fig, ax = plt.subplots(1, 2, figsize=(5, 2.5))
ax[0].imshow(images[0, 0], cmap='gray'); ax[0].set_title('original'); ax[0].axis('off')
ax[1].imshow(img, cmap='gray'); ax[1].set_title('context (targets masked)'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 2. One JEPA forward pass — loss lives in embedding space

In [ ]:
from ssllab.jepa.model import build_jepa
jepa = build_jepa(token_dim=TOKEN_DIM, n_tokens=N_TOKENS, reg_coef=0.04).to(device)
jepa.ema.to(device)
tokens = patchify(images.to(device))
loss, comp = jepa(tokens)
print('loss components (all in latent space):', {k: round(v, 4) for k, v in comp.items()})
z = jepa.embed(tokens)
print('pooled latent z:', tuple(z.shape))

## 3. Collapse diagnostics — a healthy encoder spans many directions

In [ ]:
from ssllab.eval.collapse import collapse_report
print(collapse_report(z))  # effective_rank near 1 == collapse; we want it well above 1

## 4. Next steps

Run the scripts to train for real and see the generative payoff:

```bash
python examples/jepa_basics/01_train_jepa_mnist.py --epochs 5
python examples/jepa_basics/02_linear_probe.py
python examples/generative_jepa/03_train_decoder.py --epochs 5
python examples/generative_jepa/04_train_flow_prior.py --epochs 20
python examples/generative_jepa/05_sample_and_decode.py   # -> runs/samples.png
```